In [2]:
import pandas as pd
import xarray as xr
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.iolib.smpickle import load_pickle

import comp_utils

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator
import seaborn as sns
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import pickle as pkl

# Merge the dwd hornsea nc files
dwd_Hornsea1 = xr.open_dataset("data/dwd_icon_eu_hornsea_1_20200920_20231027.nc")
dwd_Hornsea2 = xr.open_dataset("data/dwd_icon_eu_hornsea_1_20231027_20240108.nc")
dwd_Hornsea2 = dwd_Hornsea2.rename({'reference_time': 'ref_datetime', 'valid_time': 'valid_datetime'})

hornsea1_df = dwd_Hornsea1.to_dataframe()
hornsea2_df = dwd_Hornsea2.to_dataframe()


In [4]:

ncep_Hornsea1 = xr.open_dataset("data/ncep_gfs_hornsea_1_20200920_20231027.nc")
ncep_Hornsea2 = xr.open_dataset("data/ncep_gfs_hornsea_1_20231027_20240108.nc")

In [5]:
ncep_Hornsea1

<xarray.Dataset>
Dimensions:            (ref_datetime: 4526, valid_datetime: 209, latitude: 3,
                        longitude: 3)
Coordinates:
  * latitude           (latitude) float64 53.59 53.84 54.09
  * longitude          (longitude) float64 1.522 1.772 2.022
  * ref_datetime       (ref_datetime) datetime64[ns] 2020-09-20 ... 2023-10-27
  * valid_datetime     (valid_datetime) int64 0 1 2 3 4 ... 372 375 378 381 384
Data variables:
    RelativeHumidity   (ref_datetime, valid_datetime, latitude, longitude) float32 ...
    Temperature        (ref_datetime, valid_datetime, latitude, longitude) float32 ...
    WindDirection      (ref_datetime, valid_datetime, latitude, longitude) float32 ...
    WindDirection:100  (ref_datetime, valid_datetime, latitude, longitude) float32 ...
    WindSpeed          (ref_datetime, valid_datetime, latitude, longitude) float32 ...
    WindSpeed:100      (ref_datetime, valid_datetime, latitude, longitude) float32 ...

In [3]:
ncep_Hornsea2

<xarray.Dataset>
Dimensions:            (reference_time: 293, valid_time: 209, latitude: 3,
                        longitude: 3)
Coordinates:
  * latitude           (latitude) float64 53.59 53.84 54.09
  * longitude          (longitude) float64 1.522 1.772 2.022
  * reference_time     (reference_time) datetime64[ns] 2023-10-27 ... 2024-01-08
  * valid_time         (valid_time) int64 0 1 2 3 4 5 ... 372 375 378 381 384
Data variables:
    RelativeHumidity   (reference_time, valid_time, latitude, longitude) float32 ...
    Temperature        (reference_time, valid_time, latitude, longitude) float32 ...
    WindDirection      (reference_time, valid_time, latitude, longitude) float32 ...
    WindDirection:100  (reference_time, valid_time, latitude, longitude) float32 ...
    WindSpeed          (reference_time, valid_time, latitude, longitude) float32 ...
    WindSpeed:100      (reference_time, valid_time, latitude, longitude) float32 ...

In [14]:
dwd_Hornsea1

<xarray.Dataset>
Dimensions:            (ref_datetime: 4467, valid_datetime: 93, latitude: 6,
                        longitude: 6)
Coordinates:
  * latitude           (latitude) float64 53.77 53.84 53.9 53.97 54.03 54.1
  * longitude          (longitude) float64 1.702 1.767 1.832 1.897 1.962 2.027
  * ref_datetime       (ref_datetime) datetime64[ns] 2020-09-20 ... 2023-10-27
  * valid_datetime     (valid_datetime) int64 0 1 2 3 4 ... 108 111 114 117 120
Data variables:
    RelativeHumidity   (ref_datetime, valid_datetime, latitude, longitude) float32 ...
    Temperature        (ref_datetime, valid_datetime, latitude, longitude) float32 ...
    WindDirection      (ref_datetime, valid_datetime, latitude, longitude) float32 ...
    WindDirection:100  (ref_datetime, valid_datetime, latitude, longitude) float32 ...
    WindSpeed          (ref_datetime, valid_datetime, latitude, longitude) float32 ...
    WindSpeed:100      (ref_datetime, valid_datetime, latitude, longitude) float32 ...

In [16]:

dwd_Hornsea1_features = dwd_Hornsea1["WindSpeed:100"].mean(dim=["latitude","longitude"]).to_dataframe().reset_index()
dwd_Hornsea1_features["ref_datetime"] = dwd_Hornsea1_features["ref_datetime"].dt.tz_localize("UTC")
dwd_Hornsea1_features["valid_datetime"] = dwd_Hornsea1_features["ref_datetime"] + pd.TimedeltaIndex(dwd_Hornsea1_features["valid_datetime"],unit="hours")

dwd_Hornsea2_features = dwd_Hornsea2["WindSpeed:100"].mean(dim=["latitude","longitude"]).to_dataframe().reset_index()
dwd_Hornsea2_features["ref_datetime"] = dwd_Hornsea2_features["ref_datetime"].dt.tz_localize("UTC")
dwd_Hornsea2_features["valid_datetime"] = dwd_Hornsea2_features["ref_datetime"] + pd.to_timedelta(dwd_Hornsea2_features["valid_datetime"], unit='h')



In [17]:

dwd_Hornsea1_features

,ref_datetime,valid_datetime,WindSpeed:100
0,2020-09-20 00:00:00+00:00,2020-09-20 00:00:00+00:00,11.802604
1,2020-09-20 00:00:00+00:00,2020-09-20 01:00:00+00:00,11.495032
2,2020-09-20 00:00:00+00:00,2020-09-20 02:00:00+00:00,11.213223
3,2020-09-20 00:00:00+00:00,2020-09-20 03:00:00+00:00,11.135883
4,2020-09-20 00:00:00+00:00,2020-09-20 04:00:00+00:00,11.254492
...,...,...,...
415426,2023-10-27 00:00:00+00:00,2023-10-31 12:00:00+00:00,3.701586
415427,2023-10-27 00:00:00+00:00,2023-10-31 15:00:00+00:00,7.006474
415428,2023-10-27 00:00:00+00:00,2023-10-31 18:00:00+00:00,6.105711
415429,2023-10-27 00:00:00+00:00,2023-10-31 21:00:00+00:00,6.729563


In [20]:
dwd_Hornsea1_features = dwd_Hornsea1_features.drop_duplicates(subset='valid_datetime', keep='first')
dwd_Hornsea1_features = dwd_Hornsea1_features.sort_values(by='valid_datetime')

dwd_Hornsea2_features = dwd_Hornsea2_features.drop_duplicates(subset='valid_datetime', keep='first')
dwd_Hornsea2_features = dwd_Hornsea2_features.sort_values(by='valid_datetime')

